In [ ]:
# Bootstrap: agrega la raíz del repo al path e importa init_agents
# (carga .env y configura la key de tracing de OpenAI)
import sys, pathlib
_root = pathlib.Path.cwd()
while not (_root / "init_agents.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.append(str(_root))
import init_agents

## BIENVENIDOS A LA SEMANA 6

La Épica Semana Final

Y

# ¡BIENVENIDOS AL **M**ODEL **C**ONTEXT **P**ROTOCOL!

Y bienvenidos de nuevo al SDK de Agentes de OpenAI ❤️❤️❤️

### Nota:

¡Puede haber cambios con respecto al video, ya que estoy actualizando constantemente!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Para los usuarios de PC con Windows: un anuncio importante</h2>
<span style="color:#ff7800;">Tengo malas noticias. Hay un problema al ejecutar servidores MCP en PC con Windows; Mac y Linux funcionan correctamente. Este problema se conoce desde el 4 de mayo de 2025. Le pedí a o3 con Deep Research que buscara soluciones alternativas; <a href="https://chatgpt.com/share/6817bbc3-3d0c-8012-9b51-631842470628">confirmaron el problema</a> y la solución alternativa.<br/><br/>
La solución alternativa es un poco aburrida. Se trata de aprovechar "WSL", el enfoque de Microsoft para ejecutar Linux en su PC. ¡Tendrá que seguir más instrucciones de configuración! Pero es rápido, y varios estudiantes han confirmado que les funciona a la perfección. Los laboratorios de MCP de la semana 6 también funcionan. Además, WSL es una excelente manera de desarrollar software en tu PC con Windows.<br/>
Las instrucciones de configuración de WSL se encuentran en la carpeta Setup, <a href="../setup/SETUP-WSL.md">en el archivo SETUP-WSL.md</a>. Espero que esto solo te demore un poco; deberías estar listo para usar pronto. ¡Qué alegría trabajar con tecnología de vanguardia!<br/><br/>
Muchas gracias a los estudiantes Markus, Abhi, Hui-Ling y a varios otros por ayudarme a trabajar en ello y confirmar la solución.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Los imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os

In [2]:
load_dotenv(override=True)

True

### Usemos MCP en el SDK de OpenAI Agents

1. Crear un cliente

2. Crear un servidor

3. Recopilar las herramientas que el servidor puede usar

Probemos la función "Obtener mcp-server" que vimos la semana pasada

In [3]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=30) as server:
    fetch_tools = await server.list_tools()

fetch_tools

[Tool(name='fetch', description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', inputSchema={'description': 'Parameters for fetching a URL.', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'exclusiveMaximum': 1000000, 'exclusiveMinimum': 0, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer'}, 'raw': {'default': False, 'description': 'Get the actual H

## Paso de instalación adicional: si no tienes "node" en tu ordenador

La siguiente herramienta MCP usa node (el servidor Javascript) y requiere que tengas el comando "npx" instalado en tu ordenador.

Puede que ya lo tengas, pero si no, aquí tienes instrucciones muy claras sobre qué hacer, cortesía de nuestro amigo.
Y gracias al estudiante avid_learner por indicarlo.

https://chatgpt.com/share/68103af2-e2dc-8012-b259-bc135a23273b


### Y ahora, repetimos a por 2 más!

In [4]:
playwright_params = {"command": "npx","args": [ "@playwright/mcp@latest"]}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=30) as server:
    playwright_tools = await server.list_tools()

playwright_tools


[Tool(name='browser_close', description='Close the page', inputSchema={'type': 'object', 'properties': {}, 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, annotations=ToolAnnotations(title='Close browser', readOnlyHint=True, destructiveHint=False, idempotentHint=None, openWorldHint=True)),
 Tool(name='browser_resize', description='Resize the browser window', inputSchema={'type': 'object', 'properties': {'width': {'type': 'number', 'description': 'Width of the browser window'}, 'height': {'type': 'number', 'description': 'Height of the browser window'}}, 'required': ['width', 'height'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, annotations=ToolAnnotations(title='Resize browser window', readOnlyHint=True, destructiveHint=False, idempotentHint=None, openWorldHint=True)),
 Tool(name='browser_console_messages', description='Returns all console messages', inputSchema={'type': 'object', 'properties': {}, 'additi

In [5]:

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=30) as server:
    file_tools = await server.list_tools()

file_tools

[Tool(name='read_file', description="Read the complete contents of a file from the file system. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Only works within allowed directories.", inputSchema={'type': 'object', 'properties': {'path': {'type': 'string'}, 'tail': {'type': 'number', 'description': 'If provided, returns only the last N lines of the file'}, 'head': {'type': 'number', 'description': 'If provided, returns only the first N lines of the file'}}, 'required': ['path'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, annotations=None),
 Tool(name='read_multiple_files', description="Read the contents of multiple files simultaneously. This is more efficient than reading files one by one when you

### ¡Y ahora... viene el Agente con Herramientas!

In [6]:
instructions = """
Navegas por internet para completar tus instrucciones.
Eres muy capaz de navegar por internet de forma independiente para completar tu tarea, 
incluyendo aceptar todas las cookies y hacer clic en "ahora no" según corresponda 
para acceder al contenido que necesitas. Si un sitio web no te funciona, prueba con otro.
Sé persistente hasta que hayas resuelto tu tarea, probando diferentes opciones y sitios según sea necesario.
"""


async with MCPServerStdio(params=files_params, client_session_timeout_seconds=30) as mcp_server_files:
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=30) as mcp_server_browser:
        agent = Agent(
            name="investigator", 
            instructions=instructions, 
            model="gpt-4.1-mini",
            mcp_servers=[mcp_server_files, mcp_server_browser]
            )
        with trace("investigate"):
            result = await Runner.run(agent, "Encuentra una excelente receta de tarta Banoffee y luego resúmela en markdown en banoffee.md")
            print(result.final_output)


He encontrado una excelente receta de tarta Banoffee en BBC Good Food y la he resumido en formato markdown. Aquí tienes el contenido para el archivo banoffee.md:

```markdown
# Banoffee Pie Recipe

An easy family favorite with buttery pastry and sweet dulce de leche. Served with a generous dollop of cream and optionally garnished with dark chocolate.

## Ingredients

- 4 bananas, sliced
- 394g caramel or dulce de leche
- 300ml double cream
- Dark chocolate (optional, for garnish)

### For the pastry
- 100g butter, chilled (plus extra for greasing)
- 200g plain flour
- 1 medium egg, separated
- 1 tbsp golden caster sugar

## Method

1. **Make the pastry case:**
   - Put the butter and flour in a food processor and pulse until it resembles fresh breadcrumbs.
   - Add the yolk of the egg and sugar, pulse until mixed.
   - Add very cold water (1 tbsp at a time), pulsing until the dough comes together.
   - Knead gently into a smooth ball, wrap in cling film, and chill for 30 minutes.

2. *

### Revisa la traza

https://platform.openai.com/traces

### Vamos a investigar algunos marketplaces de MCP

https://mcp.so

https://glama.ai/mcp

https://smithery.ai/

https://huggingface.co/blog/LLMhacker/top-11-essential-mcp-libraries

Gran artículo de la comunidad de HuggingFace:
https://huggingface.co/blog/Kseniase/mcp



